In [0]:
# ==========================================================
# STEP 1 — Create the secret scope (run from your local terminal
# with Databricks CLI configured, NOT inside a notebook)
# ==========================================================
#
#   databricks secrets create-scope claims-snowflake
#   databricks secrets put-secret claims-snowflake snowflake-password
#       (this opens a prompt / editor — paste your Snowflake password, don't type it inline)
#
# Verify (metadata only, never shows the password):
#   databricks secrets list-scopes
#   databricks secrets list-secrets claims-snowflake


# ==========================================================
# STEP 2 — Safe notebook verification (Databricks notebook cell)
# Confirms the secret can be retrieved WITHOUT ever printing it
# ==========================================================

password = dbutils.secrets.get(
    scope="claims-snowflake",
    key="snowflake-password"
)
print("Secret retrieved successfully.")


# ==========================================================
# STEP 3 — Test Databricks -> Snowflake connectivity
# Checks auth + active session, does NOT load any project data
# ==========================================================

sf_options = {
    "sfURL": "MDHABOF-MT28339.snowflakecomputing.com",  # <-- EDIT: your Snowflake account URL
    "sfUser": "VAIBHAVX14",                        # <-- EDIT: your Snowflake username
    "sfPassword": dbutils.secrets.get(
        scope="claims-snowflake",
        key="snowflake-password"
    ),
    "sfDatabase": "HEALTHCARE_CLAIMS_DB",
    "sfSchema": "RAW",
    "sfWarehouse": "CLAIMS_WH",
    "sfRole": "ACCOUNTADMIN",
}

test_df = (
    spark.read
    .format("snowflake")
    .options(**sf_options)
    .option("query", "SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE()")
    .load()
)

display(test_df)

# Success: returns your Snowflake account, user, role, and warehouse.
# Failure: capture the exact error — a missing Snowflake Spark connector
# (check the cluster has the snowflake-jdbc + spark-snowflake libraries
# installed) or a wrong sfURL/credential are the most likely causes.
#
# IMPORTANT: never display sf_options or the password variable in output.

Secret retrieved successfully.


"""CURRENT_ACCOUNT()""","""CURRENT_USER()""","""CURRENT_ROLE()""","""CURRENT_WAREHOUSE()"""
OJ20527,VAIBHAVX14,ACCOUNTADMIN,CLAIMS_WH
